# Character CNN (Model A) on DHCD - Kaggle

Owner: Ravi. Settings: **Accelerator = GPU T4**, **Internet = On**, secret `GITHUB_TOKEN` switched on.
Run the cells one by one (Shift+Enter). The whole notebook takes about 15-30 minutes.

In [ ]:
# 1. GPU check + clone the repo on Ravi's branch
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
!git clone -q https://{token}@github.com/Nikhil04-ai/Devanagari_Word_Recognition_Project.git
%cd Devanagari_Word_Recognition_Project
!git checkout ravi-charcnn
!git log --oneline -3
!pip install -q rapidfuzz pyyaml

In [ ]:
# 2. download DHCD from the UCI repository (about 80 MB) into data/raw/dhcd (the code reads the zip directly)
!mkdir -p data/raw/dhcd
!wget -q -O data/raw/dhcd/dhcd.zip 'https://archive.ics.uci.edu/static/public/389/devanagari+handwritten+character+dataset.zip'
!ls -lh data/raw/dhcd

In [ ]:
# 3. tests (the TrOCR tests may need transformers, which Kaggle has)
!python -m pytest -q tests/test_char_cnn.py

In [ ]:
# 4. sanity check: memorise 100 training images. Must end with 'overfit test passed'
!python -m src.char_cnn.train --config configs/char_cnn/overfit.yaml

In [ ]:
# 5. full training (time budget 30 min; stops early if validation accuracy stops improving)
!python -m src.char_cnn.train --config configs/char_cnn/full.yaml
# if the session stopped: run the same line again with --resume

In [ ]:
# 6. evaluate ONCE on the test set -> results/char_cnn_full_argmax/
!python -m src.char_cnn.evaluate --ckpt checkpoints/char_cnn_full/best.pt
!cat results/char_cnn_full_argmax/metrics.json

In [ ]:
# 7. pack everything to download (Output tab / file browser on the right): results + best checkpoint
!cp checkpoints/char_cnn_full/log.csv results/char_cnn_full_argmax/training_log.csv
!zip -q -r /kaggle/working/char_cnn_outputs.zip results/char_cnn_full_argmax checkpoints/char_cnn_full/best.pt
!ls -lh /kaggle/working/char_cnn_outputs.zip